# Scenario Runs: Gas CCS and Hydrogen Storage

        This notebook demonstrates how to use `src.scenarios` helpers to:

        1. Compare dispatchable gas with CCS vs no CCS.
        2. Run the power system with reduced or zero hydrogen storage.

        Adjust the parameters in the configuration cell to explore different capacities, demand modes, and datasets.

In [1]:
        from __future__ import annotations

        import pandas as pd

        from src.demand_model import DemandMode
        from src.scenarios import ScenarioResult, compare_ccs_vs_no_ccs, low_or_zero_hydrogen_storage
        from src.units import Units as U
        

In [2]:
        # Base configuration
        renewable_capacity = 300 * U.GW
        demand_mode = DemandMode.SEASONAL
        capacity_factors_source = "era5_2024"
        enable_imports = False

        renewable_capacity, demand_mode, capacity_factors_source, enable_imports
        

(<Quantity(300, 'gigawatt')>,
 <DemandMode.SEASONAL: 'seasonal'>,
 'era5_2024',
 False)

## Gas CCS vs No CCS

        The helper below runs two scenarios with identical assumptions except for the dispatchable gas CCS capacity.

In [3]:
        ccs_results = compare_ccs_vs_no_ccs(
            renewable_capacity=renewable_capacity,
            demand_mode=demand_mode,
            capacity_factors_source=capacity_factors_source,
            enable_imports=enable_imports,
        )

        summary_rows: list[dict] = []
        for label, scenario in ccs_results.items():
            row = {
                "scenario": label,
                "energy_cost_GBP_per_MWh": scenario.energy_cost.to(U.GBP / U.MWh).magnitude,
            }

            if scenario.analysis is not None:
                row.update(
                    {
                        "minimum_medium_storage_TWh": scenario.analysis["minimum_medium_storage"].to(U.TWh).magnitude,
                        "minimum_hydrogen_storage_TWh": scenario.analysis["minimum_hydrogen_storage"].to(U.TWh).magnitude,
                        "annual_gas_ccs_energy_TWh": scenario.analysis["annual_gas_ccs_energy"].to(U.TWh).magnitude,
                        "curtailed_energy_TWh": scenario.analysis["curtailed_energy"].to(U.TWh).magnitude,
                    }
                )
                formatted = scenario.system.format_simulation_results(scenario.analysis)
            else:
                formatted = "Simulation failed: insufficient storage capacity."

            summary_rows.append(row)

            print(f"Scenario: {label}")
            print(f"Energy cost: {scenario.energy_cost:~0.2f}")
            print(formatted)
            print("-" * 80)

        pd.DataFrame(summary_rows)
        

Scenario: with_ccs
Energy cost: 62.30 GBP / MWh
- Minimum medium storage: 0.0 TWh\n- Minimum hydrogen storage:  35.3 TWh\n- DAC energy: 7.3 TWh\n- DAC CO2 removals: 11.4 Mt\n- DAC Capacity Factor: 76.0%\n- Gas CCS energy: 12.9 TWh\n- Gas CCS Capacity Factor: 13.2%\n- Curtailed energy: 263.8 TWh\n- Interconnect energy: 0.0 TWh\n
--------------------------------------------------------------------------------
Scenario: no_ccs
Energy cost: 58.94 GBP / MWh
- Minimum medium storage: 0.0 TWh\n- Minimum hydrogen storage:  19.5 TWh\n- DAC energy: 6.1 TWh\n- DAC CO2 removals: 9.6 Mt\n- DAC Capacity Factor: 63.9%\n- Gas CCS energy: 0.0 TWh\n- Gas CCS Capacity Factor: 0.0%\n- Curtailed energy: 233.2 TWh\n- Interconnect energy: 0.0 TWh\n
--------------------------------------------------------------------------------


,scenario,energy_cost_GBP_per_MWh,minimum_medium_storage_TWh,minimum_hydrogen_storage_TWh,annual_gas_ccs_energy_TWh,curtailed_energy_TWh
0,with_ccs,62.296575,0.0,35.304158,12.912182,263.752281
1,no_ccs,58.937915,0.0,19.515028,0.000000,233.210888


## Reduced and Zero Hydrogen Storage

        This section sets hydrogen storage to a very small value (1 TWh) or removes it entirely.
        Electrolyser and hydrogen generation power are also reduced/removed to keep the configuration physical.

In [4]:
        low_h2 = low_or_zero_hydrogen_storage(
            renewable_capacity=renewable_capacity,
            mode="reduced",
            demand_mode=demand_mode,
            capacity_factors_source=capacity_factors_source,
            enable_imports=enable_imports,
            reduced_storage_capacity=1.0 * U.TWh,
            reduced_electrolyser_power=5.0 * U.GW,
            reduced_hydrogen_generation_power=5.0 * U.GW,
        )

        zero_h2 = low_or_zero_hydrogen_storage(
            renewable_capacity=renewable_capacity,
            mode="zero",
            demand_mode=demand_mode,
            capacity_factors_source=capacity_factors_source,
            enable_imports=enable_imports,
        )

        h2_results = {"low_h2": low_h2, "zero_h2": zero_h2}

        h2_summary: list[dict] = []
        for label, scenario in h2_results.items():
            row = {
                "scenario": label,
                "energy_cost_GBP_per_MWh": scenario.energy_cost.to(U.GBP / U.MWh).magnitude,
            }

            if scenario.analysis is not None:
                row.update(
                    {
                        "minimum_medium_storage_TWh": scenario.analysis["minimum_medium_storage"].to(U.TWh).magnitude,
                        "minimum_hydrogen_storage_TWh": scenario.analysis["minimum_hydrogen_storage"].to(U.TWh).magnitude,
                        "annual_gas_ccs_energy_TWh": scenario.analysis["annual_gas_ccs_energy"].to(U.TWh).magnitude,
                        "curtailed_energy_TWh": scenario.analysis["curtailed_energy"].to(U.TWh).magnitude,
                    }
                )
                formatted = scenario.system.format_simulation_results(scenario.analysis)
            else:
                formatted = "Simulation failed: insufficient storage capacity."

            h2_summary.append(row)

            print(f"Scenario: {label}")
            print(f"Energy cost: {scenario.energy_cost:~0.2f}")
            print(formatted)
            print("-" * 80)

        pd.DataFrame(h2_summary)

Scenario: low_h2
Energy cost: 50.45 GBP / MWh
Simulation failed: insufficient storage capacity.
--------------------------------------------------------------------------------
Scenario: zero_h2
Energy cost: 50.03 GBP / MWh
Simulation failed: insufficient storage capacity.
--------------------------------------------------------------------------------


,scenario,energy_cost_GBP_per_MWh
0,low_h2,50.446751
1,zero_h2,50.025585


### Iterative Natural-Gas Capacity Search

    The cell below automatically re-runs the zero-hydrogen scenario while steadily increasing
    `gas_ccs_capacity` until the simulation succeeds. This makes it easy to identify the minimum
    dispatchable gas requirement (and associated energy cost / CO₂ release) needed to compensate
    for the removed hydrogen storage.

In [5]:
def find_required_gas_capacity(
        *,
        start_capacity: float = 78,
        step_capacity: float = 10,
        max_capacity: float = 200,
    ) -> pd.DataFrame:
        history: list[dict] = []
        capacity = start_capacity
        GAS_CO2_INTENSITY = 0.20196 * U.Mt_mt / U.TWh

        while capacity <= max_capacity:
            scenario = low_or_zero_hydrogen_storage(
                renewable_capacity=renewable_capacity,
                mode="zero",
                demand_mode=demand_mode,
                capacity_factors_source=capacity_factors_source,
                enable_imports=enable_imports,
                gas_ccs_capacity=capacity * U.GW,
            )

            row: dict[str, float | bool | None] = {
                "gas_capacity_GW": capacity,
                "energy_cost_GBP_per_MWh": scenario.energy_cost.to(U.GBP / U.MWh).magnitude,
                "successful": scenario.sim_df is not None and scenario.analysis is not None,
            }

            if scenario.analysis is not None:
                gas_energy = scenario.analysis["annual_gas_ccs_energy"].to(U.TWh)
                row["annual_gas_ccs_energy_TWh"] = gas_energy.magnitude
                row["estimated_residual_CO2_Mt"] = (gas_energy * GAS_CO2_INTENSITY).to(U.Mt).magnitude
            else:
                row["annual_gas_ccs_energy_TWh"] = None
                row["estimated_residual_CO2_Mt"] = None

            history.append(row)
            print(f"Tried {capacity:.0f} GW -> {'success' if row['successful'] else 'FAILED'}")

            if row["successful"]:
                break

            capacity += step_capacity

        return pd.DataFrame(history)

gas_capacity_search = find_required_gas_capacity(start_capacity=78, step_capacity=1, max_capacity=240)
gas_capacity_search

Tried 78 GW -> FAILED
Tried 79 GW -> success


,gas_capacity_GW,energy_cost_GBP_per_MWh,successful,annual_gas_ccs_energy_TWh,estimated_residual_CO2_Mt
0,78,50.025585,False,NaN,NaN
1,79,55.920271,True,17.223942,3.478547


In [6]:
from src.scenarios import find_gas_only_capacity_no_hydrogen
from src.demand_model import DemandMode

# Try zero hydrogen, ramp gas CCS from 5 GW in 5 GW steps up to 60 GW
result, gas_cap_used, residual_co2 = find_gas_only_capacity_no_hydrogen(
    renewable_capacity=300 * U.GW,
    demand_mode=DemandMode.SEASONAL,
    capacity_factors_source="era5_2024",
    enable_imports=False,
    start_capacity=78 * U.GW,
    step=1 * U.GW,
    max_capacity=120 * U.GW,
)

print(f"Gas capacity used: {gas_cap_used:~}")
print(f"Estimated residual CO2: {residual_co2:~0.2f}")
print(f"Energy cost: {result.energy_cost:~0.2f}")

#full simulation metrics:
if result.analysis:
    result.system.print_simulation_results(result.analysis)






Gas capacity used: 79 GW
Estimated residual CO2: 3.48 Mt
Energy cost: 55.92 GBP / MWh
- Minimum medium storage: 0.0 TWh\n- Minimum hydrogen storage:  0.0 TWh\n- DAC energy: 7.7 TWh\n- DAC CO2 removals: 12.1 Mt\n- DAC Capacity Factor: 80.3%\n- Gas CCS energy: 17.2 TWh\n- Gas CCS Capacity Factor: 13.2%\n- Curtailed energy: 273.9 TWh\n- Interconnect energy: 0.0 TWh\n


In [1]:
from src.scenarios import electricity_to_capture_co2
from src.scenarios import capital_cost_to_capture_co2
import src.assumptions as A
from src.units import Units as U

residual = 3.48 * U.Mt  # estimated residual CO2
electricity_needed = electricity_to_capture_co2(residual)  # TWh
cost = capital_cost_to_capture_co2(residual)  # GBP

print(cost)
print(electricity_needed)



5593777777.777778 GBP
1.5312 terawatt_hour
